# 06 - Classical ML models
Logistic Regression, Random Forest, HistGradientBoosting, XGBoost.
Needs `data/processed/chicago_domestic_minimal.csv` (notebook 05). Encoders sit inside each Pipeline, so they are fitted on **train only**.
Takes about 5-10 minutes.

In [1]:
import sys, os
from pathlib import Path
# works whether Jupyter starts in the project root or inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))      # lets us import data_utils.py from the root
print("Project root:", ROOT)

Project root: c:\Users\aru12\OneDrive\Desktop\capstone project


## 1. Imports and data

In [2]:
import joblib, numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from data_utils import *

(ROOT / "models").mkdir(exist_ok=True)
(Xtr, ytr), (Xva, yva), (Xte, yte) = load_splits()
print(f"train/val/test = {len(Xtr):,}/{len(Xva):,}/{len(Xte):,}   positive rate (train) = {ytr.mean():.3f}")
spw = (ytr == 0).sum() / (ytr == 1).sum()

train/val/test = 389,196/131,775/236,927   positive rate (train) = 0.181


## 2. Define the four models (encoders inside the Pipeline)

In [3]:
onehot = lambda: ColumnTransformer([("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                        min_frequency=30), FEATURES)])
target_enc = lambda: ColumnTransformer([("te", TargetEncoder(target_type="binary", cv=5, random_state=SEED), FEATURES)])
models = {
    "LogisticRegression": Pipeline([("p", onehot()), ("m", LogisticRegression(
        max_iter=1000, class_weight="balanced", C=1.0))]),
    "RandomForest": Pipeline([("p", target_enc()), ("m", RandomForestClassifier(
        n_estimators=200, min_samples_leaf=20, n_jobs=-1,
        class_weight="balanced_subsample", random_state=SEED))]),
    "HistGradientBoosting": Pipeline([("p", target_enc()), ("m", HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.08, l2_regularization=1.0, class_weight="balanced",
        early_stopping=True, validation_fraction=0.1, random_state=SEED))]),
    "XGBoost": Pipeline([("p", onehot()), ("m", XGBClassifier(
        n_estimators=400, max_depth=7, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, reg_lambda=2.0, scale_pos_weight=spw, tree_method="hist",
        n_jobs=-1, random_state=SEED, eval_metric="logloss"))]),
}
list(models)

['LogisticRegression', 'RandomForest', 'HistGradientBoosting', 'XGBoost']

## 3. Train. The threshold is tuned on VALIDATION; the 2025 test set is only scored

In [4]:
results = {}
for name, pipe in models.items():
    pipe.fit(Xtr, ytr)
    t = best_threshold(yva, pipe.predict_proba(Xva)[:, 1])       # tuned on validation
    p_te = pipe.predict_proba(Xte)[:, 1]                          # test: report only
    a, b = report(name, yte, p_te, t)
    results[name] = dict(acc=a, bacc=b, auc=roc_auc_score(yte, p_te), thr=t, p=p_te)
    joblib.dump(pipe, ROOT / f"models/{name}.joblib")

LogisticRegression     thr=0.53  accuracy=0.8271  balanced_acc=0.8276  gap=0.0005


c:\Users\aru12\anaconda3\envs\chicago-crime\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


RandomForest           thr=0.51  accuracy=0.8304  balanced_acc=0.8301  gap=0.0003


c:\Users\aru12\anaconda3\envs\chicago-crime\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


HistGradientBoosting   thr=0.55  accuracy=0.8296  balanced_acc=0.8299  gap=0.0003
XGBoost                thr=0.52  accuracy=0.8312  balanced_acc=0.8311  gap=0.0001


## 4. Summary table (test = all of 2025)

In [5]:
summary = pd.DataFrame({k: {"accuracy": r["acc"], "balanced_accuracy": r["bacc"], "roc_auc": r["auc"],
                            "threshold": r["thr"]} for k, r in results.items()}).T.round(4)
summary

,accuracy,balanced_accuracy,roc_auc,threshold
LogisticRegression,0.8271,0.8276,0.9094,0.53
RandomForest,0.8304,0.8301,0.9104,0.51
HistGradientBoosting,0.8296,0.8299,0.9113,0.55
XGBoost,0.8312,0.8311,0.9125,0.52


## 5. Best model: classification report + confusion matrix

In [6]:
best = max(results, key=lambda k: min(results[k]["acc"], results[k]["bacc"]))
pred = (results[best]["p"] >= results[best]["thr"]).astype(int)
print("Best model:", best)
print(classification_report(yte, pred, digits=4))
pd.DataFrame(confusion_matrix(yte, pred), index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"])

Best model: XGBoost
              precision    recall  f1-score   support

           0     0.9543    0.8313    0.8885    191750
           1     0.5371    0.8309    0.6524     45177

    accuracy                         0.8312    236927
   macro avg     0.7457    0.8311    0.7705    236927
weighted avg     0.8747    0.8312    0.8435    236927



,pred 0,pred 1
actual 0,159395,32355
actual 1,7641,37536


## 6. Leakage sanity check: with shuffled labels the AUC must be about 0.5

In [7]:
rng = np.random.default_rng(SEED)
chk = Pipeline([("p", onehot()), ("m", LogisticRegression(max_iter=300))])
chk.fit(Xtr, rng.permutation(ytr))
print(f"shuffled-label AUC on test = {roc_auc_score(yte, chk.predict_proba(Xte)[:, 1]):.3f} (should be ~0.5)")

shuffled-label AUC on test = 0.457 (should be ~0.5)
